# 02 - Behavioral Validation

Do the fear / greed / herd indicators actually relate to future returns, or are they noise? We score each against the next-day return.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))  # so 'import src...' works from notebooks/

import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config
config = load_config('../config.yaml')
ticker = config['data']['tickers'][0]
config['data']['raw_dir'] = '../data/raw'  # paths are relative to project root
ticker

In [ ]:
from src.data.loader import load_ohlcv
from src.data.cleaner import clean_ohlcv
from src.behavioral import compute_fear, compute_greed, compute_herd
from src.behavioral.validation import validate_indicator

raw = load_ohlcv(ticker, config['data']['start_date'], config['data']['end_date'],
                 raw_dir=config['data']['raw_dir'])
df = clean_ohlcv(raw)

## Build indicators and the forward return they're scored against

In [ ]:
bcfg = config['behavioral']
indicators = {
    'fear': compute_fear(df, **bcfg['fear']),
    'greed': compute_greed(df, **bcfg['greed']),
    'herd': compute_herd(df, **bcfg['herd']),
}
forward_return = df['Close'].shift(-1) / df['Close'] - 1.0

## Diagnostics

`ic` is the rank correlation with next-day return; `top`/`bottom` are the mean forward return in the highest vs lowest readings.

In [ ]:
import pandas as pd
report = {name: validate_indicator(series, forward_return) for name, series in indicators.items()}
pd.DataFrame(report).T